In [1]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import seaborn as sns
import h5py
import time
import scipy.sparse as sp
import yaml
import gc
from matplotlib import gridspec

import rmm
import cupy as cp
from rmm.allocators.cupy import rmm_cupy_allocator

import cuml
from cuml.decomposition import PCA

import psutil
import torch
from datetime import datetime

import anndata as an
import scanpy as sc
import rapids_singlecell as rsc
import scvi

sc.settings.verbosity = 3

# Enable `managed_memory`
rmm.reinitialize(
    managed_memory=True,
    pool_allocator=False,
)
cp.cuda.set_allocator(rmm_cupy_allocator)

/nfs/turbo/umms-indikar/Cooper/conda_envs/scrapids/lib/python3.12/site-packages/docrep/decorators.py:43: SyntaxWarning: 'param_categorical_covariate_keys' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)
/nfs/turbo/umms-indikar/Cooper/conda_envs/scrapids/lib/python3.12/site-packages/docrep/decorators.py:43: SyntaxWarning: 'param_continuous_covariate_keys' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)


In [2]:
start = time.perf_counter()

fpath = "/scratch/indikar_root/indikar1/shared_data/hematokytos/processed/sample_1_adata.h5ad"
adata = sc.read_h5ad(fpath)
adata.obsm['X_umap'] = adata.obsm['X_umap_X_scANVI'].copy()
sc.logging.print_memory_usage() # Log memory at the end

print(f"Total time: {time.perf_counter() - start:.2f} seconds")

adata # Return/output adata

Memory usage: current 48.79 GB, difference +48.79 GB
Total time: 56.65 seconds


/nfs/turbo/umms-indikar/Cooper/conda_envs/scrapids/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


AnnData object with n_obs × n_vars = 1125041 × 52164
    obs: 'soma_joinid', 'dataset_id', 'assay', 'assay_ontology_term_id', 'cell_type', 'cell_type_ontology_term_id', 'development_stage', 'development_stage_ontology_term_id', 'disease', 'disease_ontology_term_id', 'donor_id', 'is_primary_data', 'observation_joinid', 'self_reported_ethnicity', 'self_reported_ethnicity_ontology_term_id', 'sex', 'sex_ontology_term_id', 'suspension_type', 'tissue', 'tissue_ontology_term_id', 'tissue_type', 'tissue_general', 'tissue_general_ontology_term_id', 'raw_sum', 'nnz', 'raw_mean_nnz', 'raw_variance_nnz', 'n_measured_vars', 'basename', 'dataset_id_int', 'n_counts', 'n_genes', '_scvi_batch', '_scvi_labels', 'leiden'
    var: 'n_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'X_umap_X_scANVI', 'X_umap_raw_data', 'X_umap_scVI', '_scvi_manager_uuid', '_scvi_uuid', 'basename_colors', 'basename_palette

# HSC Compartment

In [3]:
list(adata.obs['basename'].unique())

['fibroblasts',
 'mesenchymal_cells',
 'myeloid_cells',
 'lymphoid_cells',
 'hematopoietic_progenitors',
 'innate_lymphoid_cells',
 'hsc',
 'endothelial_cells']

In [4]:
# Subset to HSCs
query = ['hsc', 'hematopoietic_progenitors']
# query = ['hematopoietic_progenitors']
# # query = ['hsc']
bdata = adata[adata.obs['basename'].isin(query), :].copy()
bdata.obs_names_make_unique()

n_duplicates = bdata.obs.index.duplicated().sum()
print(f"Dropped {n_duplicates} duplicate obs_names.")

bdata = bdata[~bdata.obs.index.duplicated(keep='first')].copy()
rsc.get.anndata_to_GPU(adata) # move to GPU
# Show shape
print(f"Subset shape: {bdata.shape}")

# Cluster with Leiden
rsc.get.anndata_to_CPU(bdata) # move to CPU
rsc.get.anndata_to_GPU(bdata) # move to GPU
rsc.pp.neighbors(bdata, n_neighbors=100, use_rep='X_scANVI')
rsc.tl.leiden(bdata, resolution=0.4, key_added='hsc_leiden')
bdata.obs['hsc_clusters'] = bdata.obs['hsc_leiden'].astype(int).apply(lambda x: f"H{x+1}")
bdata.obs['hsc_clusters'] = bdata.obs['hsc_clusters'].astype(str)

# Show cluster counts
print("\nCluster counts:")
print(bdata.obs['hsc_clusters'].value_counts())

# Show obs columns
print("\nAvailable .obs columns:")
print(bdata.obs.columns.tolist())

# Show unique cell types
print("\nUnique cell types:")
print(bdata.obs['cell_type'].unique())

# Return the AnnData object (Jupyter will pretty-print it)
bdata

/nfs/turbo/umms-indikar/Cooper/conda_envs/scrapids/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Dropped 0 duplicate obs_names.
Subset shape: (109819, 52164)
[2025-04-22 16:52:00.535] [CUML] [debug] n_neighbors=100
[2025-04-22 16:52:00.535] [CUML] [debug] Calling knn graph run
[2025-04-22 16:52:00.535] [CUML] [debug] Done. Calling fuzzy simplicial set
[2025-04-22 16:52:00.641] [CUML] [debug] Done. Calling remove zeros

Cluster counts:
hsc_clusters
H4     12886
H15    12459
H16     9199
H2      8461
H7      8423
H10     7626
H1      7390
H14     6888
H9      5879
H11     4660
H8      4533
H3      4464
H5      4014
H18     3795
H17     3760
H6      2416
H13     1818
H12     1148
Name: count, dtype: int64

Available .obs columns:
['soma_joinid', 'dataset_id', 'assay', 'assay_ontology_term_id', 'cell_type', 'cell_type_ontology_term_id', 'development_stage', 'development_stage_ontology_term_id', 'disease', 'disease_ontology_term_id', 'donor_id', 'is_primary_data', 'observation_joinid', 'self_reported_ethnicity', 'self_reported_ethnicity_ontology_term_id', 'sex', 'sex_ontology_term_id',

AnnData object with n_obs × n_vars = 109819 × 52164
    obs: 'soma_joinid', 'dataset_id', 'assay', 'assay_ontology_term_id', 'cell_type', 'cell_type_ontology_term_id', 'development_stage', 'development_stage_ontology_term_id', 'disease', 'disease_ontology_term_id', 'donor_id', 'is_primary_data', 'observation_joinid', 'self_reported_ethnicity', 'self_reported_ethnicity_ontology_term_id', 'sex', 'sex_ontology_term_id', 'suspension_type', 'tissue', 'tissue_ontology_term_id', 'tissue_type', 'tissue_general', 'tissue_general_ontology_term_id', 'raw_sum', 'nnz', 'raw_mean_nnz', 'raw_variance_nnz', 'n_measured_vars', 'basename', 'dataset_id_int', 'n_counts', 'n_genes', '_scvi_batch', '_scvi_labels', 'leiden', 'hsc_leiden', 'hsc_clusters'
    var: 'n_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'X_umap_X_scANVI', 'X_umap_raw_data', 'X_umap_scVI', '_scvi_manager_uuid', '_scvi_uuid', 'basena

In [5]:
del adata
gc.collect()
sc.logging.print_memory_usage()

Memory usage: current 56.57 GB, difference +7.78 GB


# Map cell type abbreviations

In [6]:
cell_type_map = {
    'erythroid progenitor cell': 'EryP',
    'megakaryocyte progenitor cell': 'MkP',
    'lymphoid lineage restricted progenitor cell': 'LLP',
    'basophil mast progenitor cell': 'BaMP',
    'common dendritic progenitor': 'CDP',
    'megakaryocyte-erythroid progenitor cell': 'MEP',
    'hematopoietic multipotent progenitor cell': 'MPP',
    'granulocyte monocyte progenitor cell': 'GMP',
    'early lymphoid progenitor': 'ELP',
    'common myeloid progenitor': 'CMP',
    'hematopoietic stem cell': 'HSC',
    'hematopoietic cell': 'HC',
    'hematopoietic precursor cell': 'HPC',
    'cord blood hematopoietic stem cell': 'HSC (cord blood)',
    'CD34-positive, CD38-negative hematopoietic stem cell': 'HSC (CD34+CD38−)',
}

bdata.obs['cell_type_abbrev'] = bdata.obs['cell_type'].map(cell_type_map)
bdata.obs['cell_type_abbrev'] = bdata.obs['cell_type_abbrev'].astype(str)

print(bdata.obs['cell_type_abbrev'].value_counts(dropna=False).to_string())

cell_type_abbrev
EryP                23229
MPP                 23176
HSC                 13464
HC                  11837
HSC (cord blood)     9056
MEP                  7196
LLP                  5401
HPC                  4253
GMP                  3608
ELP                  2690
CMP                  2595
CDP                  1873
MkP                   904
BaMP                  463
HSC (CD34+CD38−)       74


In [7]:

# PCA on the scANVI embedding
pca = PCA(n_components=7)  
bdata.obsm['temp'] = pca.fit_transform(bdata.obsm['X_scANVI'])

bdata

AnnData object with n_obs × n_vars = 109819 × 52164
    obs: 'soma_joinid', 'dataset_id', 'assay', 'assay_ontology_term_id', 'cell_type', 'cell_type_ontology_term_id', 'development_stage', 'development_stage_ontology_term_id', 'disease', 'disease_ontology_term_id', 'donor_id', 'is_primary_data', 'observation_joinid', 'self_reported_ethnicity', 'self_reported_ethnicity_ontology_term_id', 'sex', 'sex_ontology_term_id', 'suspension_type', 'tissue', 'tissue_ontology_term_id', 'tissue_type', 'tissue_general', 'tissue_general_ontology_term_id', 'raw_sum', 'nnz', 'raw_mean_nnz', 'raw_variance_nnz', 'n_measured_vars', 'basename', 'dataset_id_int', 'n_counts', 'n_genes', '_scvi_batch', '_scvi_labels', 'leiden', 'hsc_leiden', 'hsc_clusters', 'cell_type_abbrev'
    var: 'n_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'X_umap_X_scANVI', 'X_umap_raw_data', 'X_umap_scVI', '_scvi_manager_uuid', '

In [8]:
bdata.obsm['temp'].shape

(109819, 7)

In [9]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)

In [ ]:
rsc.tl.tsne(
    bdata, 
    use_rep='X_scANVI',
    perplexity=15,
    early_exaggeration=20,
)

plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = 4.5, 5

sc.pl.tsne(
    bdata,
    color=['cell_type', 'hsc_clusters'],
    size=10,
    ncols=1,
    alpha=1,
    add_outline=True,
    outline_color=('k', 'k'),
    palette='tab20c',
    title=["", ""],
    frameon=False,
    na_in_legend=False,
)


In [ ]:
# … your existing data prep …
X = pd.crosstab(bdata.obs['tissue'], bdata.obs['cell_type'])

cell_type_map = {
    'erythroid progenitor cell': 'EryP',
    'megakaryocyte progenitor cell': 'MkP',
    'lymphoid lineage restricted progenitor cell': 'LLP',
    'basophil mast progenitor cell': 'BaMP',
    'common dendritic progenitor': 'CDP',
    'megakaryocyte-erythroid progenitor cell': 'MEP',
    'hematopoietic multipotent progenitor cell': 'MPP',
    'granulocyte monocyte progenitor cell': 'GMP',
    'early lymphoid progenitor': 'ELP',
    'common myeloid progenitor': 'CMP',
    'hematopoietic stem cell': 'HSC',
    'hematopoietic cell': 'HC',
    'hematopoietic precursor cell': 'HPC',
    'cord blood hematopoietic stem cell': 'HSC (cord blood)',
    'CD34-positive, CD38-negative hematopoietic stem cell': 'HSC (CD34+CD38−)',
}

for k, v in cell_type_map.items():
    print(f"\item {v}: {k}")

X.columns = X.columns.map(cell_type_map)
X = X[sorted(X.columns)]

threshold = 50
X_filtered = X[X.sum(axis=1) > threshold].T

# compute totals
col_totals = np.log1p(X_filtered.sum(axis=0))
row_totals = np.log1p(X_filtered.sum(axis=1))

n_cols = len(col_totals)
n_rows = len(row_totals)

# positions at cell centers
col_pos = np.arange(n_cols) + 0.5
row_pos = np.arange(n_rows) + 0.5

# figure + GridSpec
fig = plt.figure(figsize=(8, 5))
gs = gridspec.GridSpec(2, 2,
              width_ratios=[4, 1],
              height_ratios=[1, 4],
              wspace=0.02,
              hspace=0.02)

# 1) Top: column sums
ax_col = fig.add_subplot(gs[0, 0])
ax_col.bar(
    col_pos, col_totals.values,
    width=0.55, align='center',
    color='gray',
    ec='k',
)
ax_col.set_xlim(0, n_cols)
ax_col.set_xticks([])              # no x‑ticks on the barplot
ax_col.set_ylabel("cells (log)")

# 2) Left: heatmap with ticks
ax_heat = fig.add_subplot(gs[1, 0])
sns.heatmap(
    np.log1p(X_filtered),
    cmap='plasma',
    cbar=False,
    ax=ax_heat,
    linewidths=1
)
# explicitly set heatmap ticks at the same positions
ax_heat.set_xticks(col_pos)
ax_heat.set_xticklabels(col_totals.index, ha='center')
ax_heat.set_yticks(row_pos)
ax_heat.set_yticklabels(row_totals.index, rotation=0)
ax_heat.set_xlabel("")
ax_heat.set_ylabel("")

# capture exact limits
xlim = ax_heat.get_xlim()
ylim = ax_heat.get_ylim()
ax_col.set_xlim(xlim)

# 3) Right: row sums
ax_row = fig.add_subplot(gs[1, 1])
ax_row.barh(
    row_pos, row_totals.values,
    height=0.55, align='center',
    color='gray',
    ec='k',
)
ax_row.set_ylim(0, n_rows)
ax_row.set_yticks([])             # no y‑ticks on the barplot
ax_row.set_xlabel("cells (log)")
# invert so it matches heatmap orientation
ax_row.invert_yaxis()
ax_row.set_ylim(ylim)

sns.despine(ax=ax_row)
sns.despine(ax=ax_col)

plt.tight_layout()
plt.show()